# UdaciSense: Model Compression - Google Colab Pro

**🚀 GPU-accelerated compression with Google Colab Pro**

**Before starting:**
1. Enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU (T4)
2. Ensure your baseline model is available from the previous notebook
3. Update the `DRIVE_PROJECT_PATH` below

**Expected time: ~30-45 minutes with T4 GPU**

## Compression Targets:
- **70% model size reduction** (5.96 MB → 1.79 MB)
- **60% inference speedup** 
- **<5% accuracy drop** (maintain >83% from 87.4% baseline)

## Step 1: Google Drive Setup

In [ ]:
# Cell 1: Mount Google Drive and navigate to project
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where you uploaded your project
# DRIVE_PROJECT_PATH = '/content/drive/MyDrive/udacity-projects/udaci-model-optimization/project/starter_kit'
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/udacity-ml-compression-pipeline/project/starter_kit'

# Navigate to project directory
os.chdir(DRIVE_PROJECT_PATH)
print(f"✅ Changed to directory: {os.getcwd()}")

# Add project root to Python path for imports
project_root = os.path.abspath('../..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"✅ Added to Python path: {project_root}")

# Verify project structure
required_dirs = ['src', 'notebooks', 'models', 'results']
missing_dirs = []
for d in required_dirs:
    if os.path.exists(d):
        print(f"✅ Found: {d}/")
    else:
        missing_dirs.append(d)
        print(f"❌ Missing: {d}/")

if missing_dirs:
    print(f"\n⚠️ Please upload these directories to your Google Drive: {missing_dirs}")
else:
    print("\n🎉 All required directories found!")

In [ ]:
# Cell 2: Install requirements in Colab using UV (faster)
!curl -LsSf https://astral.sh/uv/install.sh | sh
!source ~/.bashrc && uv --version

# Install packages using UV (much faster than pip)
!~/.cargo/bin/uv pip install --system torch>=2.0.0 torchvision>=0.15.0 
!~/.cargo/bin/uv pip install --system matplotlib seaborn pandas scikit-learn pillow tqdm plotly
!~/.cargo/bin/uv pip install --system thop  # For FLOPs calculation

print("✅ All packages installed with UV (faster than pip)!")

## Step 2: Verify GPU Setup

In [ ]:
# Cell 3: Check GPU availability
import torch
import warnings
warnings.filterwarnings('ignore')

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🚀 GPU Available: {gpu_name}")
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print("⚠️ No GPU found. Please enable GPU in Runtime → Change runtime type")

print(f"Device: {device}")

# Additional devices for different techniques
cpu_device = torch.device('cpu')  # For quantized models
gpu_device = device if torch.cuda.is_available() else cpu_device  # For training

## Step 3: Import Project Modules

In [ ]:
# Cell 4: Import all required libraries
%load_ext autoreload
%autoreload 2

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import time
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset

# Setup Python path for imports
import sys
import os
print(f"📂 Current working directory: {os.getcwd()}")
print(f"📂 Contents: {[f for f in os.listdir('.') if not f.startswith('.')]}")

# Add the current directory (starter_kit) to Python path so 'src' and internal imports work
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
    print(f"✅ Added to Python path: {current_dir}")

# Also add the src directory so internal module imports work (utils, etc.)
src_dir = os.path.join(current_dir, 'src')
if os.path.exists(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)
    print(f"✅ Added src to Python path: {src_dir}")

# Import compression modules
try:
    from src.compression.post_training.pruning import prune_model
    from src.compression.post_training.quantization import quantize_model
    from src.compression.post_training.graph_optimization import optimize_model, verify_model_equivalence
    from src.compression.in_training.distillation import train_with_distillation, MobileNetV3_Household_Small
    from src.compression.in_training.pruning import train_with_pruning
    from src.compression.in_training.quantization import train_model_qat, QuantizableMobileNetV3_Household
    print("✅ Compression modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    
    # For Colab: if src imports fail, check directory structure
    if 'src' not in os.listdir('.'):
        print("❌ 'src' directory not found in current directory")
        print("💡 Please ensure you've run Cell 1 (Google Drive mount) first")
        print("💡 And verify the DRIVE_PROJECT_PATH is correct")
        print("📂 Current directory contents:", os.listdir('.'))
        raise ImportError("src directory not found - please check Google Drive setup")
    else:
        print("✅ src directory exists, re-raising import error")
        raise

# Import utility modules
try:
    from src.utils import MAX_ALLOWED_ACCURACY_DROP, TARGET_INFERENCE_SPEEDUP, TARGET_MODEL_COMPRESSION
    from src.utils.data_loader import get_household_loaders, get_input_size, print_dataloader_stats, visualize_batch
    from src.utils.model import MobileNetV3_Household, load_model, save_model, print_model_summary
    from src.utils.visualization import plot_multiple_models_comparison
    from src.utils.compression import (
        compare_experiments, compare_optimized_model_to_baseline, evaluate_optimized_model, list_experiments,
        is_quantized
    )
    from src.utils.evaluation import evaluate_model_metrics
    print("✅ Utils modules imported successfully")
    
    # Print optimization targets
    print(f"\n🎯 Optimization Targets:")
    print(f"   Max accuracy drop: {MAX_ALLOWED_ACCURACY_DROP*100}%")
    print(f"   Target speedup: {TARGET_INFERENCE_SPEEDUP*100}%")
    print(f"   Target compression: {TARGET_MODEL_COMPRESSION*100}%")
    
except ImportError as e:
    print(f"❌ Utils import error: {e}")
    print("💡 Make sure Cell 1 (Google Drive setup) was run successfully")
    print("💡 And verify all src/ subdirectories were uploaded to Google Drive")
    raise

In [ ]:
# Cell 5: Set random seed for reproducibility
def set_deterministic_mode(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    return seed_worker

seed_worker = set_deterministic_mode(42)
g = torch.Generator()
g.manual_seed(42)

print("✅ Reproducibility mode set (seed=42)")

## Step 4: Setup Directories and Load Data

In [ ]:
# Cell 6: Create directories for compression techniques
compression_types = [
    "post_training/pruning",
    "post_training/quantization", 
    "post_training/graph_optimization",
    "in_training/distillation",
    "in_training/quantization",
    "in_training/pruning",
]

for comp_type in compression_types:
    models_dir = f"models/{comp_type}"
    models_ckp_dir = f"{models_dir}/checkpoints"
    results_dir = f"results/{comp_type}"
    
    os.makedirs(models_ckp_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

print("📁 Created compression experiment directories")

In [ ]:
# Cell 7: Load household objects dataset (GPU optimized)
GPU_BATCH_SIZE = 256 if torch.cuda.is_available() else 128
NUM_WORKERS = 2  # Colab works best with 2 workers

print(f"🔄 Loading dataset (batch_size={GPU_BATCH_SIZE}, num_workers={NUM_WORKERS})...")

train_loader, test_loader = get_household_loaders(
    image_size="CIFAR", 
    batch_size=GPU_BATCH_SIZE, 
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# Get input size and class information
input_size = get_input_size("CIFAR")
class_names = train_loader.dataset.classes

print(f"\n📊 Dataset loaded with {len(class_names)} classes")
print(f"Input size: {input_size}")

# Print dataset stats
for name, loader in [('Train', train_loader), ('Test', test_loader)]:
    print(f"\n{name} set statistics:")
    print_dataloader_stats(loader, name.lower())

## Step 5: Load Baseline Model and Metrics

In [ ]:
# Cell 8: Load baseline model and metrics from previous notebook
print("🔍 Loading baseline model and metrics...")

# Try to load from Colab results first
baseline_model_name = "baseline_mobilenet_colab"
baseline_model_path = f"models/{baseline_model_name}/checkpoints/model.pth"
baseline_metrics_path = f"results/{baseline_model_name}/metrics.json"

# Check if baseline model exists
if os.path.exists(baseline_model_path):
    print(f"✅ Found baseline model: {baseline_model_path}")
    baseline_model = load_model(baseline_model_path, gpu_device)
    baseline_model.eval()
else:
    # Fallback to original baseline
    baseline_model_name = "baseline_mobilenet"
    baseline_model_path = f"models/{baseline_model_name}/checkpoints/model.pth"
    baseline_metrics_path = f"results/{baseline_model_name}/metrics.json"
    
    if os.path.exists(baseline_model_path):
        print(f"✅ Found original baseline model: {baseline_model_path}")
        baseline_model = load_model(baseline_model_path, gpu_device)
        baseline_model.eval()
    else:
        print("❌ No baseline model found!")
        print("Please run the baseline notebook first to generate the baseline model")
        raise FileNotFoundError("Baseline model not found")

# Load baseline metrics
if os.path.exists(baseline_metrics_path):
    with open(baseline_metrics_path, 'r') as f:
        baseline_metrics = json.load(f)
    print("✅ Loaded baseline metrics")
else:
    print("⚠️ Baseline metrics not found - generating new metrics...")
    # Generate metrics for baseline model
    baseline_metrics = evaluate_model_metrics(
        baseline_model, test_loader, gpu_device, len(class_names), class_names, input_size,
        save_path=baseline_metrics_path
    )
    print("✅ Generated and saved baseline metrics")

# Display baseline performance
print(f"\n📊 BASELINE PERFORMANCE:")
print(f"   🎯 Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"   📏 Model Size: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"   ⏱️ CPU Inference: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms")
if 'cuda' in baseline_metrics.get('timing', {}):
    print(f"   ⚡ GPU Inference: {baseline_metrics['timing']['cuda']['avg_time_ms']:.2f} ms")

# Calculate targets
target_model_size = baseline_metrics['size']['model_size_mb'] * (1 - TARGET_MODEL_COMPRESSION)
target_cpu_time = baseline_metrics['timing']['cpu']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
min_accuracy = baseline_metrics['accuracy']['top1_acc'] * (1 - MAX_ALLOWED_ACCURACY_DROP)

print(f"\n🎯 OPTIMIZATION TARGETS:")
print(f"   Target size: {target_model_size:.2f} MB ({TARGET_MODEL_COMPRESSION*100}% reduction)")
print(f"   Target CPU time: {target_cpu_time:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100}% speedup)")
print(f"   Min accuracy: {min_accuracy:.2f}% (max {MAX_ALLOWED_ACCURACY_DROP*100}% drop)")

## Step 6: Compression Techniques

Now we'll implement and evaluate different compression techniques. Each technique will be optimized for GPU when applicable.

### 6.1 Post-Training Quantization

In [ ]:
# Cell 9: Post-training quantization (optimized for Colab)
def apply_post_training_quantization(quantization_type, backend, device):
    """
    Apply quantization to a model with given method and backend.
    
    Args:
        quantization_type: Quantization method ("static" or "dynamic")
        backend: Backend for quantization ("fbgemm" for x86 or "qnnpack" for ARM)
        device: Which device to use for model loading, training, and evaluation
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Define unique experiment name
    experiment_name = f"post_training/quantization/{quantization_type}"
    
    # Create experiment subdirectories
    os.makedirs(f"models/{experiment_name}", exist_ok=True)
    os.makedirs(f"results/{experiment_name}", exist_ok=True)
    
    print(f"🔧 Applying {quantization_type} quantization with {backend} backend")
    
    # Make a copy of the baseline model and move to CPU for quantization
    orig_model = load_model(baseline_model_path, cpu_device)
    orig_model.eval()
    
    # Apply post-training quantization
    quantized_model = quantize_model(
        orig_model,
        quantization_type=quantization_type,
        calibration_data_loader=train_loader if quantization_type == "static" else None,
        calibration_num_batches=5 if quantization_type == "static" else None,
        backend=backend,
    )
    
    # Save the quantized model
    save_model(quantized_model, f"models/{experiment_name}/model.pth")
    
    # Verify model is quantized
    is_quantized(quantized_model)
    
    # Evaluate on CPU (quantized models run on CPU)
    evaluate_optimized_model(
        quantized_model, test_loader, experiment_name, class_names, input_size, device=cpu_device
    )
    
    # Compare with baseline
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        quantized_model,
        experiment_name,
        test_loader,
        class_names,
        device=cpu_device,
    )
    
    return quantized_model, comparison_results, experiment_name

# Apply dynamic quantization (fastest and most compatible)
quantization_type = "dynamic"  # "dynamic" or "static"
backend = "fbgemm"  # "fbgemm" for x86 CPUs
device_quantization = cpu_device  # Quantization works on CPU

print("🚀 Starting post-training quantization...")
quantized_model, quantized_comparison_results, quantization_experiment = apply_post_training_quantization(
    quantization_type, backend, device_quantization
)
print(f"✅ Quantization completed: {quantization_experiment}")

### 6.2 Post-Training Pruning

In [ ]:
# Cell 10: Post-training pruning (GPU optimized)
def apply_post_training_pruning(config):
    """
    Apply post-training pruning to a model with given pruning method and amount
    
    Args:
        config: Dictionary containing the configuration for the experiment
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Extract parameters
    amount, pruning_method, device_pruning = config['amount'], config['pruning_method'], config['device']
    
    # Define experiment name
    experiment_name = f"post_training/pruning/{pruning_method}_{amount}_{device_pruning.type}"
    experiment_name = experiment_name.replace('.', '-')
    
    # Create directories
    os.makedirs(f"models/{experiment_name}", exist_ok=True)
    os.makedirs(f"results/{experiment_name}", exist_ok=True)
    
    print(f"✂️ Applying post-training pruning: {pruning_method} {amount:.1%}")
    
    # Load model on specified device
    orig_model = load_model(baseline_model_path, device_pruning)
    orig_model.eval()
    
    # Apply pruning
    pruned_model = prune_model(
        orig_model, 
        pruning_method, 
        amount, 
        config["modules_to_prune"], 
        config["custom_pruning_fn"]
    )
    
    # Save model
    save_model(pruned_model, f"models/{experiment_name}/model.pth")
    
    # Evaluate
    evaluate_optimized_model(
        pruned_model, test_loader, experiment_name, class_names, input_size, device=device_pruning
    )
    
    # Compare with baseline
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        pruned_model,
        experiment_name,
        test_loader,
        class_names,
        device=device_pruning,
    )
    
    return pruned_model, comparison_results, experiment_name

# Configuration for pruning (can use GPU for faster evaluation)
pruning_config = {
    'pruning_method': "magnitude",  # L1 magnitude-based pruning
    'amount': 0.3,  # 30% pruning
    'modules_to_prune': None,  # Auto-detect Conv2d and Linear layers
    'n': None,
    'dim': None,
    'custom_pruning_fn': None,
    'device': gpu_device,  # Use GPU for evaluation
}

print("🚀 Starting post-training pruning...")
pruned_model, pruned_comparison_results, pruning_experiment = apply_post_training_pruning(pruning_config)
print(f"✅ Pruning completed: {pruning_experiment}")

### 6.3 Knowledge Distillation (GPU Accelerated)

In [ ]:
# Cell 11: Knowledge distillation (GPU optimized training)
def apply_knowledge_distillation(teacher_model, student_model, config):
    """
    Apply knowledge distillation from a teacher model to a student model.
    
    Args:
        teacher_model: Pre-trained teacher model
        student_model: Smaller student model to train
        config: Dictionary containing the training configuration for the experiment
        
    Returns:
        Tuple of (distilled_student_model, comparison_results, experiment_name)
    """
    # Extract parameters
    temperature, alpha = config['temperature'], config['alpha']
    num_epochs = config['num_epochs']
    device_training = config['device']
    
    # Define experiment name
    experiment_name = f"in_training/distillation/temp{temperature}_alpha{alpha}_epochs{num_epochs}"
    experiment_name = experiment_name.replace('.', '-')
    
    # Create directories
    os.makedirs(f"models/{experiment_name}", exist_ok=True)
    os.makedirs(f"results/{experiment_name}", exist_ok=True)
    
    print(f"🎓 Applying knowledge distillation: T={temperature}, α={alpha}")
    if torch.cuda.is_available():
        print(f"   Expected time: ~10-15 minutes on T4 GPU")
    
    # Move models to training device
    teacher_model = teacher_model.to(device_training)
    student_model = student_model.to(device_training)
    teacher_model.eval()  # Teacher stays in eval mode
    
    # Train student with distillation
    distilled_model, distillation_stats, best_accuracy, best_epoch = train_with_distillation(
        student_model,
        teacher_model,
        train_loader,
        test_loader,
        config,
        checkpoint_path=f"models/{experiment_name}/model.pth"
    )
    
    # Save training stats
    with open(f"results/{experiment_name}/training_stats.json", 'w') as f:
        json.dump(distillation_stats, f, indent=4)
    
    # Save model
    save_model(distilled_model, f"models/{experiment_name}/model.pth")
    
    # Evaluate
    evaluate_optimized_model(
        distilled_model, 
        test_loader, 
        experiment_name,
        class_names,
        input_size,
        is_in_training_technique=True,
        training_stats=distillation_stats,
        device=device_training,
    )
    
    # Compare with baseline
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        distilled_model,
        experiment_name,
        test_loader,
        class_names,
        device=device_training,
    )
    
    return distilled_model, comparison_results, experiment_name

# Create teacher and student models
teacher_model = baseline_model  # Use trained baseline as teacher
student_model = MobileNetV3_Household_Small(num_classes=len(class_names))  # Smaller student

print(f"📊 Teacher model parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")
print(f"📊 Student model parameters: {sum(p.numel() for p in student_model.parameters()):,}")
compression_ratio = sum(p.numel() for p in student_model.parameters()) / sum(p.numel() for p in teacher_model.parameters())
print(f"📊 Student is {compression_ratio:.1%} the size of teacher")

# GPU-optimized distillation config
distillation_config = {
    'num_epochs': 20,  # Fewer epochs with GPU
    'criterion': nn.CrossEntropyLoss(),
    'optimizer': optim.AdamW(student_model.parameters(), lr=0.001, weight_decay=1e-4),
    'scheduler': optim.lr_scheduler.StepLR(
        optim.AdamW(student_model.parameters(), lr=0.001, weight_decay=1e-4), 
        step_size=6, gamma=0.1
    ),
    'alpha': 0.7,  # 70% distillation loss, 30% hard targets
    'temperature': 4.0,  # Temperature for softmax
    'patience': 5,  # Early stopping
    'device': gpu_device,  # Use GPU for training
}

print("🚀 Starting knowledge distillation...")
distilled_model, distilled_comparison_results, distillation_experiment = apply_knowledge_distillation(
    teacher_model, student_model, distillation_config
)
print(f"✅ Distillation completed: {distillation_experiment}")

### 6.4 Quantization-Aware Training (GPU Accelerated)

In [ ]:
# Cell 11.5: Define QAT config and validation (CRITICAL - Run before QAT training)
# Create model and config first (needed for validation)
qat_model = QuantizableMobileNetV3_Household(quantize=False)

# Create initial optimizer and scheduler for QAT function
qat_optimizer = optim.AdamW(qat_model.parameters(), lr=0.001, weight_decay=1e-4)
qat_scheduler = optim.lr_scheduler.StepLR(qat_optimizer, step_size=6, gamma=0.1)

qat_config = {
    'qat_start_epoch': 5, 
    'freeze_bn_epochs': 3, 
    'num_epochs': 20,
    'criterion': nn.CrossEntropyLoss(),
    'optimizer': qat_optimizer,  # Required by current function
    'scheduler': qat_scheduler,  # Required by current function
    'optimizer_class': optim.AdamW,  # For recreation after QAT activation
    'optimizer_kwargs': {'lr': 0.001, 'weight_decay': 1e-4},
    'scheduler_class': optim.lr_scheduler.StepLR,
    'scheduler_kwargs': {'step_size': 6, 'gamma': 0.1},
    'patience': 5, 
    'device': gpu_device, 
    'device_for_inference': cpu_device,
    'grad_clip_norm': 1.0,
    'num_calibration_batches': 100,
}

backend = "fbgemm"  # For x86 CPUs

print("🛡️ VALIDATING QAT SETUP TO PREVENT GPU TIME WASTE")
print("=" * 60)

# Force reload the module path and import
import sys
import os
import importlib

# Add current directory to Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Looking for validation file in: {current_dir}")
print(f"File exists: {os.path.exists('validate_qat_setup.py')}")

# Force import/reload
try:
    import validate_qat_setup
    importlib.reload(validate_qat_setup)
    from validate_qat_setup import validate_qat_complete_setup, quick_validate
    print("✅ Validation module imported successfully!")
except Exception as e:
    print(f"❌ Import failed: {e}")
    print("Files in current directory:")
    print([f for f in os.listdir('.') if f.endswith('.py')])
    raise

# Step 1: Quick validation (30 seconds)
print("\n⚡ STEP 1: Quick validation...")
quick_errors = quick_validate(qat_config)
if quick_errors:
    print("❌ Quick validation failed:")
    for error in quick_errors:
        print(f"   {error}")
    raise Exception("❌ STOP: Fix errors before proceeding with expensive GPU training!")
else:
    print("✅ Quick validation passed!")

# Step 2: Comprehensive validation (2-3 minutes)
print("\n🔬 STEP 2: Comprehensive validation (simulates full QAT workflow)...")
print("This takes 2-3 minutes but saves 15+ minutes if there are issues!")

errors, warnings = validate_qat_complete_setup(
    qat_config, train_loader, test_loader, backend
)

# Final decision
if errors:
    print(f"\n❌ VALIDATION FAILED - {len(errors)} critical errors found:")
    for i, error in enumerate(errors, 1):
        print(f"   {i}. {error}")
    print("\n🚨 DO NOT PROCEED - Training will crash and waste GPU time!")
    raise Exception("Validation failed - fix errors before training")
else:
    print(f"\n🎉 VALIDATION PASSED!")
    print("✅ All 13 validation tests successful")
    print("🚀 Safe to proceed with QAT training!")
    
    if warnings:
        print(f"\n⚠️ {len(warnings)} warnings (non-critical):")
        for warning in warnings:
            print(f"   • {warning}")
        print("Consider addressing warnings for optimal performance")

print("\n" + "="*60)
print("🎯 VALIDATION COMPLETE - Ready for QAT training!")
print("="*60)

In [ ]:
# Cell 12: Quantization-aware training (GPU optimized)
def apply_quantization_aware_training(model, config, backend):
    """Apply quantization-aware training to a model."""
    # Extract parameters
    qat_start_epoch, num_epochs = config['qat_start_epoch'], config['num_epochs']

    # Define experiment name
    experiment_name = f"in_training/quantization/epochs{num_epochs}_start{qat_start_epoch}"
    experiment_name = experiment_name.replace('.', '-')

    # Create directories
    os.makedirs(f"models/{experiment_name}", exist_ok=True)
    os.makedirs(f"results/{experiment_name}", exist_ok=True)

    print(f"🔧 Applying QAT: start epoch {qat_start_epoch}, total {num_epochs}")
    if torch.cuda.is_available():
        print(f"   Expected time: ~15-20 minutes on T4 GPU")

    # Move model to training device
    model = model.to(config['device'])

    # Train with QAT
    quantized_model, qat_stats, qat_best_accuracy, qat_best_epoch = train_model_qat(
        model,
        train_loader,
        test_loader,
        config,
        checkpoint_path=f"models/{experiment_name}/checkpoints/model.pth",
        backend=backend,
    )

    # Save model and stats
    with open(f"results/{experiment_name}/training_stats.json", 'w') as f:
        json.dump(qat_stats, f, indent=4)
    save_model(quantized_model, f"models/{experiment_name}/model.pth")

    # Evaluate and compare
    evaluate_optimized_model(
        quantized_model, 
        test_loader,
        experiment_name, 
        class_names, 
        input_size,
        is_in_training_technique=True, 
        training_stats=qat_stats,
        device=config["device_for_inference"],
    )

    comparison_results = compare_optimized_model_to_baseline(
        baseline_model, 
        quantized_model, 
        experiment_name,
        test_loader, 
        class_names,
        device=config["device_for_inference"],
    )

    return quantized_model, comparison_results, experiment_name

# Use the qat_model, qat_config, and backend defined in previous cell
print("🚀 Starting quantization-aware training...")
qat_model_trained, qat_comparison_results, qat_experiment = apply_quantization_aware_training(
    qat_model, 
    qat_config, 
    backend
)
print(f"✅ QAT completed: {qat_experiment}")

## Step 7: Compare All Techniques

In [ ]:
# Cell 13: Compare all compression techniques
print("📊 Comparing all compression techniques...")

# List all completed experiments
experiments = list_experiments()
print(f"\n🧪 Found {len(experiments)} completed experiments:")
for exp in experiments:
    print(f"   • {exp}")

if experiments:
    # Generate comprehensive comparison
    comparison_df = compare_experiments(
        experiments=experiments,
        baseline_metrics=baseline_metrics
    )
    
    # Display results
    print("\n📈 COMPRESSION RESULTS SUMMARY:")
    print(comparison_df.to_string(index=False))
    
    # Calculate which techniques meet targets
    print("\n🎯 TARGET ACHIEVEMENT:")
    print(f"   Size target: <{target_model_size:.2f} MB")
    print(f"   Speed target: <{target_cpu_time:.2f} ms (CPU)")
    print(f"   Accuracy target: >{min_accuracy:.2f}%")
    
    # Check which experiments meet all targets
    meets_all_targets = []
    if 'model_size_mb' in comparison_df.columns:
        for idx, row in comparison_df.iterrows():
            size_ok = row['model_size_mb'] <= target_model_size
            acc_ok = row['top1_acc'] >= min_accuracy
            speed_ok = row.get('cpu_avg_time_ms', float('inf')) <= target_cpu_time
            
            if size_ok and acc_ok and speed_ok:
                meets_all_targets.append(row['experiment_name'])
    
    if meets_all_targets:
        print(f"\n✅ Experiments meeting ALL targets:")
        for exp in meets_all_targets:
            print(f"   🏆 {exp}")
    else:
        print(f"\n⚠️ No single technique meets all targets")
        print(f"   💡 Consider combining techniques in a multi-stage pipeline")
else:
    print("\n⚠️ No completed experiments found")
    print("Run at least one compression technique above first")

## Step 8: Analysis and Next Steps

In [ ]:
# Cell 14: Generate analysis and recommendations
print("📋 COMPRESSION ANALYSIS SUMMARY")
print("=" * 50)

# Target summary
print(f"\n🎯 CTO REQUIREMENTS:")
print(f"   Model Size: {baseline_metrics['size']['model_size_mb']:.2f} MB → {target_model_size:.2f} MB ({TARGET_MODEL_COMPRESSION*100}% reduction)")
print(f"   Inference Speed: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms → {target_cpu_time:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100}% speedup)")
print(f"   Accuracy: Keep above {min_accuracy:.2f}% (max {MAX_ALLOWED_ACCURACY_DROP*100}% drop from {baseline_metrics['accuracy']['top1_acc']:.2f}%)")

# Technique effectiveness summary
print(f"\n📊 TECHNIQUE EFFECTIVENESS:")

techniques_summary = {
    "Dynamic Quantization": {
        "size_reduction": "~75%",
        "speed_improvement": "High (INT8 ops)", 
        "accuracy_impact": "Low (~1-2%)",
        "gpu_time": "N/A (CPU only)"
    },
    "Magnitude Pruning": {
        "size_reduction": "30% (sparsity)",
        "speed_improvement": "Medium (sparse support needed)",
        "accuracy_impact": "Low-Medium (~2-3%)",
        "gpu_time": "Fast evaluation"
    },
    "Knowledge Distillation": {
        "size_reduction": "High (smaller architecture)",
        "speed_improvement": "High (fewer parameters)",
        "accuracy_impact": "Medium (depends on student size)",
        "gpu_time": "10-15 min training"
    },
    "Quantization-Aware Training": {
        "size_reduction": "~75%",
        "speed_improvement": "High (INT8 ops)",
        "accuracy_impact": "Lower than post-training",
        "gpu_time": "15-20 min training"
    }
}

for technique, metrics in techniques_summary.items():
    print(f"\n   {technique}:")
    print(f"      Size: {metrics['size_reduction']}")
    print(f"      Speed: {metrics['speed_improvement']}")
    print(f"      Accuracy: {metrics['accuracy_impact']}")
    print(f"      GPU Time: {metrics['gpu_time']}")

# Multi-stage pipeline recommendation
print(f"\n🔄 MULTI-STAGE PIPELINE RECOMMENDATION:")
print(f"   Stage 1: Magnitude Pruning (30% sparsity)")
print(f"            → Removes redundant weights")
print(f"            → Expected: ~30% size reduction, minimal accuracy loss")
print(f"   ")
print(f"   Stage 2: Dynamic Quantization (FP32→INT8)")
print(f"            → Reduces precision of remaining weights")
print(f"            → Expected: Additional ~75% reduction of pruned model")
print(f"   ")
print(f"   Combined: ~77% total size reduction (exceeds 70% target)")
print(f"             Significant speed improvement (exceeds 60% target)")
print(f"             Accuracy loss manageable with fine-tuning")

print(f"\n🚀 NEXT STEPS:")
print(f"   1. Implement multi-stage pipeline in 03_pipeline_colab_pro.ipynb")
print(f"   2. Apply pruning → quantization sequentially")
print(f"   3. Fine-tune at each stage to recover accuracy")
print(f"   4. Validate final model meets all CTO requirements")

# Save analysis results
analysis_results = {
    'baseline_metrics': baseline_metrics,
    'targets': {
        'size_mb': target_model_size,
        'cpu_time_ms': target_cpu_time,
        'min_accuracy': min_accuracy
    },
    'experiments_completed': experiments,
    'pipeline_recommendation': [
        "magnitude_pruning_30_percent",
        "dynamic_quantization_int8",
        "optional_fine_tuning"
    ]
}

with open('results/compression_analysis.json', 'w') as f:
    json.dump(analysis_results, f, indent=4)

print(f"\n💾 Analysis saved to results/compression_analysis.json")
print(f"📁 All results automatically synced to Google Drive!")

## 🎉 Compression Analysis Complete!

**Your compression techniques have been successfully evaluated with GPU acceleration!**

### 📊 What You Achieved:
- ✅ **GPU Training**: Significantly faster than CPU-only compression
- ✅ **Multiple Techniques**: Tested quantization, pruning, distillation, and QAT
- ✅ **Performance Analysis**: Comprehensive comparison of all methods
- ✅ **Pipeline Strategy**: Clear roadmap for meeting CTO targets

### 🚀 Next Steps:
1. **Implement Multi-Stage Pipeline**: Use `03_pipeline_colab_pro.ipynb`
2. **Combine Best Techniques**: Apply pruning + quantization sequentially 
3. **Meet All Targets**: 70% size reduction + 60% speedup + <5% accuracy drop

### 💾 Your Files (Auto-synced to Drive):
- `models/*/` - Compressed models from each technique
- `results/*/` - Performance metrics and comparisons
- `results/compression_analysis.json` - Summary and recommendations

**Ready for the multi-stage pipeline! 🎯**